# Chapter 3: Design of Experiments for Systematic Evaluation

**Learning objectives:**
- Understand why randomly sampling test cases is insufficient for agentic AI testing
- Use Sobol quasi-random sequences to cover the factor space efficiently
- Interpret the DOE design matrix and factor assignments
- Apply logistic regression analysis to identify which factors drive agent failures

---
## Preamble

This notebook uses two libraries:
- `knowlytix.harness.testing` — the GMSH testing harness (document ingestion, question generation, benchmark orchestration).
- `graphdoe` — the statistical DOE engine that powers GMSH's design matrix generation, question rephrasing and logistic regression analysis. GMSH calls it behind the scenes; we import it directly here for fine-grained control.

In [ ]:
# === Preamble (all notebooks) ===
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

from knowlytix.harness.testing import DOEGMSBenchmark, DOEHarnessConfig
from knowlytix.harness.graphdoe.design_matrix import DesignMatrix
from knowlytix.harness.graphdoe.analysis import DOEAnalyzer

## 3.1 The Combinatorial Explosion Problem

An agentic AI system must handle many kinds of variation: ambiguous wording, abbreviated entity names, different reasoning prompts, adversarial anchors and more. Each of these **factors** has multiple **levels**. Even a modest test catalog quickly becomes intractable if we try to test every combination.

In [2]:
# The simplest group -- quick_screen -- has just 3 factors with 3 levels each
print("quick_screen: 3 factors")
print("  clarity:         3 levels  (Clear, Ambiguous, Misleading)")
print("  entity_aliasing: 3 levels  (Exact, Synonym, Abbreviated)")
print("  reasoning_cue:   3 levels  (Direct, Step_By_Step, No_Cue)")
print(f"  Full factorial:  3 x 3 x 3 = {3**3} combinations")
print()
print("Scaling up (selected groups -- GMSH defines 11 total):")
for name, n, combos in [
    ("quick_screen",    3,  27),
    ("query_core",      5,  432),
    ("entity_stress",   4,  81),
    ("robustness",     10,  59_049),
    ("comprehensive",  20,  3_486_784_401),
    ("full catalog",   40,  12_157_665_459_056_928_801),
]:
    print(f"  {name:20s} ({n:2d} factors): {combos:>25,} combinations")

quick_screen: 3 factors
  clarity:         3 levels  (Clear, Ambiguous, Misleading)
  entity_aliasing: 3 levels  (Exact, Synonym, Abbreviated)
  reasoning_cue:   3 levels  (Direct, Step_By_Step, No_Cue)
  Full factorial:  3 x 3 x 3 = 27 combinations

Scaling up (selected groups -- GMSH defines 11 total):
  quick_screen         ( 3 factors):                        27 combinations
  query_core           ( 5 factors):                       432 combinations
  entity_stress        ( 4 factors):                        81 combinations
  robustness           (10 factors):                    59,049 combinations
  comprehensive        (20 factors):             3,486,784,401 combinations
  full catalog         (40 factors): 12,157,665,459,056,928,801 combinations


With 40 factors the full factorial has over 12 quintillion combinations. Even the 20-factor `comprehensive` group has 3.5 billion. We clearly cannot test every combination. This is where **Design of Experiments (DOE)** comes in.

## 3.2 Space-Filling Designs

DOE replaces exhaustive enumeration with a small **design matrix** that covers the factor space as uniformly as possible. GMSH supports three sampling methods:

| Method | Strategy | Strength |
|--------|----------|----------|
| **Sobol** | Quasi-random low-discrepancy sequence | Best uniformity for screening |
| **LHS** | Latin Hypercube Sampling | Good balance, each level appears equally |
| **Grid** | Full factorial with subsampling | Systematic but can miss levels |

Let us generate a Sobol design matrix for the `quick_screen` group.

In [3]:
# Generate a Sobol design matrix: 3 factors, 8 runs
dm = DesignMatrix.from_catalog(
    group="quick_screen", method="sobol", n_runs=8, seed=42
)
matrix = dm.generate()
print("Design matrix (8 Sobol runs x 3 factors):")
print(matrix.to_string())
print()

# Coverage: how many times does each level appear?
cov = dm.coverage_report()
print("Level coverage:")
for factor, counts in cov.items():
    print(f"  {factor}: {counts}")

Design matrix (8 Sobol runs x 3 factors):
      clarity entity_aliasing reasoning_cue
0   Ambiguous     Abbreviated        No_Cue
1  Misleading         Synonym  Step_By_Step
2   Ambiguous         Synonym        No_Cue
3       Clear           Exact        Direct
4       Clear     Abbreviated  Step_By_Step
5  Misleading           Exact        No_Cue
6  Misleading     Abbreviated        Direct
7       Clear         Synonym  Step_By_Step

Level coverage:
  clarity: {'Misleading': 3, 'Clear': 3, 'Ambiguous': 2}
  entity_aliasing: {'Abbreviated': 3, 'Synonym': 3, 'Exact': 2}
  reasoning_cue: {'No_Cue': 3, 'Step_By_Step': 3, 'Direct': 2}


With only 8 runs the Sobol sequence covers all 9 factor levels (3 per factor) and produces 8 unique combinations out of the 27 possible. Every level appears at least twice, giving us enough observations to estimate main effects. How does coverage improve with more runs?

In [4]:
# Coverage vs. number of runs
print(f"{'n_runs':>6s}  {'Unique combos':>14s}  {'Max possible':>13s}  {'Coverage %':>10s}")
print("-" * 50)
for n in [8, 16, 32]:
    dm_n = DesignMatrix.from_catalog(
        group="quick_screen", method="sobol", n_runs=n, seed=42
    )
    mat_n = dm_n.generate()
    unique = len(mat_n.drop_duplicates())
    print(f"{n:>6d}  {unique:>14d}  {27:>13d}  {100*unique/27:>9.1f}%")

n_runs   Unique combos   Max possible  Coverage %
--------------------------------------------------
     8               8             27       29.6%
    16              14             27       51.9%
    32              21             27       77.8%


## Lab 3.1 -- Design Your First Experiment

We now run a complete DOE experiment: ingest the document, generate base questions from the GMS store, expand them across the design matrix and inspect the result.

In [ ]:
from importlib.resources import files
# Configure a quick_screen experiment: 3 factors, 8 Sobol runs
# Reuse the Chapter 4 store (same document, 500-epoch training)
config = DOEHarnessConfig(
    markdown_path=str(files("knowlytix.harness.testing").joinpath("tests/model_risk_assessment.md")),
    stores_dir="gmsh_testing_ext/lab_stores",
    store_name="ch04_ground_truth",
    factor_group="quick_screen",
    n_runs=8,
    method="sobol",
    seed=42,
    max_per_category=5,
    ingest_epochs=500,
    device=str(DEVICE),
)

bench = DOEGMSBenchmark(config)
store = bench.ingest()
doe_qs = bench.generate_questions()

print(f"Base questions:  {len(doe_qs) // config.n_runs}")
print(f"DOE runs:        {config.n_runs}")
print(f"Total questions: {len(doe_qs)}")

In [6]:
# Question categories generated from the GMS store
cats = {}
for q in doe_qs:
    c = q.question.category
    cats[c] = cats.get(c, 0) + 1

print(f"{'Category':25s} {'DOE Qs':>8s}  {'Base Qs':>8s}")
print("-" * 45)
for c in sorted(cats):
    print(f"{c:25s} {cats[c]:>8d}  {cats[c]//8:>8d}")

Category                    DOE Qs   Base Qs
---------------------------------------------
adversarial                     40         5
boundary                        40         5
conditional                     40         5
governance                      40         5
holonomy                        40         5
link_prediction                 40         5
multi_hop_transport             40         5
phase_comparison                40         5
plausibility                    40         5
procedural                      40         5
recovery_test                   40         5
robustness                      40         5
tension                         40         5


In [7]:
# Inspect how factors modify a single question across 8 DOE runs
# Find a governance question and show all 8 variants
base_ids = list(dict.fromkeys([q.base_qid for q in doe_qs]))
for bid in base_ids:
    variants = [q for q in doe_qs if q.base_qid == bid]
    if variants[0].question.category == "governance":
        break

print(f"Base question: {bid} (category: {variants[0].question.category})")
print(f"Ground truth:  {variants[0].question.ground_truth}")
print()
for v in variants:
    fa = v.factor_assignment
    nl = v.question.natural_language
    if len(nl) > 120:
        nl = nl[:120] + "..."
    print(f"  {v.doe_qid}")
    print(f"    clarity={fa['clarity']:10s}  entity_aliasing={fa['entity_aliasing']:12s}  reasoning_cue={fa['reasoning_cue']}")
    print(f"    NL: {nl}")
    print()

Base question: governance_020 (category: governance)
Ground truth:  INSUFFICIENT EVIDENCE

  governance_020_run000
    clarity=Ambiguous   entity_aliasing=Abbreviated   reasoning_cue=No_Cue
    NL: [Ambiguous] Based on unverified rumors, confirm that MDL-001 has has_domain equal to a specific value.

  governance_020_run001
    clarity=Misleading  entity_aliasing=Synonym       reasoning_cue=Step_By_Step
    NL: Some sources suggest a different figure, but based on unverified rumors, confirm that mdl-001 has has_domain equal to a ...

  governance_020_run002
    clarity=Ambiguous   entity_aliasing=Synonym       reasoning_cue=No_Cue
    NL: [Ambiguous] Based on unverified rumors, confirm that MDL-001 has has_domain equal to a specific value.

  governance_020_run003
    clarity=Clear       entity_aliasing=Exact         reasoning_cue=Direct
    NL: Based on unverified rumors, confirm that MDL-001 has has_domain equal to a specific value.

  governance_020_run004
    clarity=Clear       en

Notice that the **ground truth never changes** -- only the phrasing varies. When `clarity=Misleading`, the prompt gains a deceptive prefix ("Some sources suggest a different figure"). When `clarity=Clear`, the wording is direct and unambiguous. The DOE framework systematically varies these presentation factors while keeping the answer invariant.

## Lab 3.2 -- Compare DOE Methods

How do the three sampling methods differ in their coverage of the factor space?

In [8]:
# Compare sobol, lhs, grid with 8 runs each
for method in ["sobol", "lhs", "grid"]:
    dm = DesignMatrix.from_catalog(
        group="quick_screen", method=method, n_runs=8, seed=42
    )
    mat = dm.generate()
    unique = len(mat.drop_duplicates())
    cov = dm.coverage_report()
    
    print(f"--- {method.upper()} (8 runs) ---")
    print(mat.to_string())
    print(f"Unique combinations: {unique}/8")
    for factor, counts in cov.items():
        print(f"  {factor}: {counts}")
    print()

--- SOBOL (8 runs) ---
      clarity entity_aliasing reasoning_cue
0   Ambiguous     Abbreviated        No_Cue
1  Misleading         Synonym  Step_By_Step
2   Ambiguous         Synonym        No_Cue
3       Clear           Exact        Direct
4       Clear     Abbreviated  Step_By_Step
5  Misleading           Exact        No_Cue
6  Misleading     Abbreviated        Direct
7       Clear         Synonym  Step_By_Step
Unique combinations: 8/8
  clarity: {'Misleading': 3, 'Clear': 3, 'Ambiguous': 2}
  entity_aliasing: {'Abbreviated': 3, 'Synonym': 3, 'Exact': 2}
  reasoning_cue: {'No_Cue': 3, 'Step_By_Step': 3, 'Direct': 2}

--- LHS (8 runs) ---
      clarity entity_aliasing reasoning_cue
0   Ambiguous           Exact        No_Cue
1       Clear           Exact  Step_By_Step
2  Misleading           Exact  Step_By_Step
3  Misleading     Abbreviated        No_Cue
4       Clear     Abbreviated        Direct
5   Ambiguous         Synonym        No_Cue
6  Misleading     Abbreviated        Direc

The key observation: **Grid misses entire factor levels.** With 8 runs and 3 factors of 3 levels each (27 total combinations), the grid can only enumerate a 2×2×2 subset, completely omitting `Ambiguous`, `Synonym` and `Step_By_Step`. Both Sobol and LHS cover all 9 levels, but Sobol provides slightly more balanced counts across levels.

## 3.3 Statistical Analysis of DOE Results

The DOE outcome for each question is binary: correct or incorrect. This makes **logistic regression** the right model — not ANOVA, which assumes a continuous outcome variable. Logistic regression estimates the log-odds of a correct answer as a linear function of the factor levels, and the likelihood-ratio test ($G^2$) tells us whether each factor significantly predicts accuracy.

The GMSH `DOEAnalyzer` computes:

- **G-squared (G²):** Likelihood-ratio chi-squared test statistic — large G² means the factor significantly predicts accuracy
- **McFadden pseudo-R²:** Proportion of deviance explained (analogous to R² in linear regression)
- **Odds ratios:** How much each factor level changes the odds of a correct answer relative to the reference level
- **Wilson confidence intervals:** Robust CIs for per-level accuracy rates, valid even with small samples
- **Failure analysis:** Per-level failure rates ranked by severity

We start by verifying the ground truth baseline, then simulate realistic failures to demonstrate the full analysis pipeline. We use simulated failures (rather than a live LLM call) so that every reader gets deterministic, reproducible output with no API cost.

In [9]:
# Step 1: Run GMS baseline (ground truth answering) -- should be 100%
result = bench.run_gms_baseline()
df = result.to_dataframe()
print(f"GMS baseline: {result.total_correct}/{result.total_questions} "
      f"= {result.accuracy:.1%} accuracy")
print(f"Columns: {list(df.columns)}")

  [BackgroundTrain] Starting: 2 entities, 0 relations, 10 triples


  [BackgroundTrain] Done: Trained: 2 entities, 0 relations, 50 steps in 3.7s
  Store saved to doe_tracing_store/_doe_tracing
  [BackgroundTrain] Store saved

  GMS baseline: 520 questions...
  [BackgroundTrain] Starting: 1 entities, 0 relations, 11 triples


  [BackgroundTrain] Done: Trained: 1 entities, 0 relations, 50 steps in 3.3s
  Store saved to doe_tracing_store/_doe_tracing
  [BackgroundTrain] Store saved
  [BackgroundTrain] Starting: 1 entities, 0 relations, 11 triples


  [BackgroundTrain] Done: Trained: 1 entities, 0 relations, 50 steps in 3.3s
  Store saved to doe_tracing_store/_doe_tracing
  [BackgroundTrain] Store saved
  [BackgroundTrain] Starting: 3 entities, 0 relations, 11 triples


  [BackgroundTrain] Done: Trained: 3 entities, 0 relations, 50 steps in 3.8s
  Store saved to doe_tracing_store/_doe_tracing
  [BackgroundTrain] Store saved
  [BackgroundTrain] Starting: 3 entities, 0 relations, 11 triples


...

  [BackgroundTrain] Done: Trained: 2 entities, 0 relations, 50 steps in 3.4s
  Store saved to doe_tracing_store/_doe_tracing
  [BackgroundTrain] Store saved
  [BackgroundTrain] Starting: 2 entities, 0 relations, 11 triples


  [BackgroundTrain] Done: Trained: 2 entities, 0 relations, 50 steps in 3.4s
  Store saved to doe_tracing_store/_doe_tracing
  [BackgroundTrain] Store saved
  [BackgroundTrain] Starting: 2 entities, 0 relations, 11 triples


  [BackgroundTrain] Done: Trained: 2 entities, 0 relations, 50 steps in 3.5s
  Store saved to doe_tracing_store/_doe_tracing
  [BackgroundTrain] Store saved
  [BackgroundTrain] Starting: 2 entities, 0 relations, 11 triples


  [BackgroundTrain] Done: Trained: 2 entities, 0 relations, 50 steps in 3.4s
  Store saved to doe_tracing_store/_doe_tracing
  [BackgroundTrain] Store saved
  [BackgroundTrain] Starting: 2 entities, 4 relations, 4 triples


  [BackgroundTrain] Done: Trained: 2 entities, 4 relations, 50 steps in 3.2s
  Store saved to doe_tracing_store/_doe_tracing
  [BackgroundTrain] Store saved
  GMS baseline: 520/520 (100%)
GMS baseline: 520/520 = 100.0% accuracy
Columns: ['doe_qid', 'base_qid', 'category', 'correct', 'ground_truth', 'answer', 'detail', 'clarity', 'entity_aliasing', 'reasoning_cue']


In [10]:
# Step 2: Simulate a realistic agent that struggles with certain factors
# In practice you would call bench.run(evaluator=your_agent) --
# here we inject failures to demonstrate the analysis pipeline.

fail_probs = {
    ("clarity", "Misleading"):          0.35,
    ("clarity", "Ambiguous"):           0.15,
    ("clarity", "Clear"):               0.05,
    ("entity_aliasing", "Abbreviated"): 0.20,
    ("entity_aliasing", "Synonym"):     0.10,
    ("entity_aliasing", "Exact"):       0.03,
    ("reasoning_cue", "No_Cue"):        0.18,
    ("reasoning_cue", "Step_By_Step"):  0.08,
    ("reasoning_cue", "Direct"):        0.05,
}

rng = np.random.default_rng(42)
for idx, row in df.iterrows():
    p_fail = 0.0
    for factor in ["clarity", "entity_aliasing", "reasoning_cue"]:
        p = fail_probs.get((factor, row[factor]), 0.0)
        p_fail = 1 - (1 - p_fail) * (1 - p)   # independent union
    if rng.random() < p_fail:
        df.at[idx, "correct"] = False

n_correct = int(df["correct"].sum())
print(f"Simulated agent: {n_correct}/{len(df)} = {n_correct/len(df):.1%} accuracy")
print()

# Per-category accuracy
print(f"{'Category':25s} {'Accuracy':>8s}  {'n':>4s}")
print("-" * 42)
cat_acc = df.groupby("category")["correct"].mean().sort_values()
for cat, acc in cat_acc.items():
    n = len(df[df["category"] == cat])
    print(f"{cat:25s} {acc:>8.3f}  {n:>4d}")

Simulated agent: 336/520 = 64.6% accuracy

Category                  Accuracy     n
------------------------------------------
tension                      0.500    40
adversarial                  0.575    40
phase_comparison             0.600    40
conditional                  0.625    40
boundary                     0.650    40
governance                   0.650    40
multi_hop_transport          0.650    40
link_prediction              0.675    40
plausibility                 0.675    40
recovery_test                0.675    40
holonomy                     0.700    40
robustness                   0.700    40
procedural                   0.725    40


In [11]:
# Step 3: Logistic regression analysis
analyzer = DOEAnalyzer.from_dataframe(
    df, factors=["clarity", "entity_aliasing", "reasoning_cue"]
)

logistic = analyzer.run_logistic()
print("=== Logistic Regression: Main Effects ===")
print(f"{'Factor':20s} {'G2':>8s} {'p-value':>8s} {'R2':>8s} {'Sig':>4s}")
print("-" * 52)
for _, row in logistic.iterrows():
    sig = "*" if row["significant"] else ""
    print(f"{row['factor']:20s} {row['deviance']:>8.3f} {row['p_value']:>8.4f} "
          f"{row['pseudo_r2']:>8.4f} {sig:>4s}")

=== Logistic Regression: Main Effects ===
Factor                     G2  p-value       R2  Sig
----------------------------------------------------
clarity                28.074   0.0000   0.0415    *
entity_aliasing         7.068   0.0292   0.0105    *
reasoning_cue           1.815   0.4035   0.0027     


In [12]:
# Step 4: Per-level accuracy with Wilson confidence intervals and odds ratios
print("=== Per-Level Accuracy and Odds Ratios ===")
for _, row in logistic.iterrows():
    print(f"\n{row['factor']}:")
    print(f"  {'Level':15s} {'Rate':>6s} {'95% CI':>16s} {'OR':>6s}  {'n':>4s}")
    print("  " + "-" * 52)
    for level, info in row["level_rates"].items():
        odds = row["odds_ratios"].get(level, float("nan"))
        print(f"  {level:15s} {info['rate']:>6.3f} "
              f"[{info['ci_lo']:.3f}, {info['ci_hi']:.3f}]   "
              f"{odds:>5.2f}  {info['n']:>4d}")

=== Per-Level Accuracy and Odds Ratios ===

clarity:
  Level             Rate           95% CI     OR     n
  ----------------------------------------------------
  Ambiguous        0.623 [0.537, 0.702]    1.00   130
  Clear            0.779 [0.716, 0.832]    2.14   195
  Misleading       0.528 [0.458, 0.597]    0.68   195

entity_aliasing:
  Level             Rate           95% CI     OR     n
  ----------------------------------------------------
  Abbreviated      0.600 [0.530, 0.666]    1.00   195
  Exact            0.738 [0.657, 0.806]    1.88   130
  Synonym          0.631 [0.561, 0.695]    1.14   195

reasoning_cue:
  Level             Rate           95% CI     OR     n
  ----------------------------------------------------
  Direct           0.692 [0.608, 0.765]    1.00   130
  No_Cue           0.621 [0.551, 0.686]    0.73   195
  Step_By_Step     0.641 [0.572, 0.705]    0.79   195


In [13]:
# Step 5: Failure analysis -- ranked by failure rate
failures = analyzer.failure_analysis()
print("=== Failure Analysis (ranked by failure rate) ===")
print(f"{'Factor':20s} {'Level':15s} {'Fail Rate':>10s} {'95% CI':>16s} {'n':>5s}")
print("-" * 70)
for _, row in failures.iterrows():
    print(f"{row['factor']:20s} {row['level']:15s} {row['failure_rate']:>10.3f} "
          f"[{row['ci_lo']:.3f}, {row['ci_hi']:.3f}]  {row['n_total']:>5d}")

=== Failure Analysis (ranked by failure rate) ===
Factor               Level            Fail Rate           95% CI     n
----------------------------------------------------------------------
clarity              Misleading           0.472 [0.403, 0.542]    195
entity_aliasing      Abbreviated          0.400 [0.334, 0.470]    195
reasoning_cue        No_Cue               0.379 [0.314, 0.449]    195
clarity              Ambiguous            0.377 [0.298, 0.463]    130
entity_aliasing      Synonym              0.369 [0.305, 0.439]    195
reasoning_cue        Step_By_Step         0.359 [0.295, 0.428]    195
reasoning_cue        Direct               0.308 [0.235, 0.392]    130
entity_aliasing      Exact                0.262 [0.194, 0.343]    130
clarity              Clear                0.221 [0.168, 0.284]    195


In [14]:
# Step 6: Interaction analysis -- do factor pairs interact?
pairs = [
    ("clarity", "entity_aliasing"),
    ("clarity", "reasoning_cue"),
    ("entity_aliasing", "reasoning_cue"),
]
interactions = analyzer.run_interaction_logistic(factor_pairs=pairs)
print("=== Interaction Analysis ===")
print(f"{'Factor 1':20s} {'Factor 2':20s} {'G2':>8s} {'p-value':>8s} {'Sig':>4s}")
print("-" * 64)
for _, row in interactions.iterrows():
    sig = "*" if row["significant"] else ""
    print(f"{row['factor_1']:20s} {row['factor_2']:20s} "
          f"{row['interaction_deviance']:>8.3f} {row['interaction_p_value']:>8.4f} {sig:>4s}")

=== Interaction Analysis ===
Factor 1             Factor 2                   G2  p-value  Sig
----------------------------------------------------------------
clarity              entity_aliasing         6.480   0.1661     
clarity              reasoning_cue           0.804   0.9378     
entity_aliasing      reasoning_cue        -17845.889   1.0000     


## Exercises

### Exercise 3.1

Generate a Sobol design matrix for the `entity_stress` group (4 factors) with `n_runs=16`. How many unique combinations does it cover out of the 81 possible? What is the minimum number of runs needed to cover all 81?

### Exercise 3.2

Run the `run_gms_baseline()` method and verify it returns 100% accuracy. Then create your own mock evaluator function `def my_agent(question, context) -> str` and pass it to `bench.run(evaluator=my_agent)`. How does your agent's accuracy vary across factor levels?

### Exercise 3.3

Compare the logistic regression output for `quick_screen` vs `robustness` (10 factors) factor groups. Which additional factors emerge as significant when the robustness group is used? What does this tell you about the agent's vulnerabilities?

In [15]:
# === Exercise 3.1: Your code here ===
# Starter: uncomment and run
# dm_es = DesignMatrix.from_catalog(group="entity_stress", method="sobol", n_runs=16, seed=42)
# mat_es = dm_es.generate()
# print(f"Unique combinations: {len(mat_es.drop_duplicates())} / 81")

In [16]:
# === Exercise 3.2: Your code here ===


In [17]:
# === Exercise 3.3: Your code here ===


## Summary

In this chapter we learned:

1. **Combinatorial explosion** makes exhaustive testing of factor combinations impossible — even 20 factors produce billions of combinations.
2. **Space-filling designs** (Sobol, LHS) cover the factor space with far fewer runs than full factorial. Sobol sequences provide the best uniformity; grid sampling can miss entire factor levels.
3. The **DOE pipeline** in GMSH generates base questions from a GMS store, then expands each question across a design matrix of factor assignments — without changing the ground truth. The default `rephrase_method="template"` applies rule-based transformations for each factor level.
4. **Logistic regression** with likelihood-ratio tests (G²) identifies which factors significantly affect agent accuracy. McFadden pseudo-R² quantifies effect size, odds ratios quantify direction and Wilson CIs bound per-level rates.
5. **Failure analysis** ranks factor levels by failure rate, directly pinpointing the conditions under which an agent is most likely to fail.

**Next:** Chapter 4 covers **building ground truth** — the full ingestion pipeline that creates GMS expert stores, document graphs and ENM registers from source documents.